**Loading Llama3.2 model**

In [ ]:
!pip install -U langchain-community mypy_extensions
!pip install -U ddgs
!pip install langchain langchain-groq langchain-classic
!pip install python-dotenv
!pip install -U sentence-transformers faiss-cpu

In [2]:
import os
import numpy as np
import pandas as pd
from langchain_groq import ChatGroq
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
llm=ChatGroq( model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=1024,
)

Generating response for the prompt

In [3]:
def build_llama3_prompt(messages):
    prompt = "<|begin_of_text|>"
    for m in messages:
        prompt+= f"<|start_header_id|>{m['role']}<|end_header_id|>\n\n{m['content']}<|eot_id|>"
    prompt+= "<|start_header_id|>assistant<|end_header_id|>\n\n"
    return prompt


def generate(messages, max_new_tokens=200, do_sample=True, temperature=1, top_p=0.25):
    prompt = build_llama3_prompt(messages)
    return llm.invoke(
        prompt,
        max_tokens=max_new_tokens,
        temperature=temperature if do_sample else 0.0,
        top_p=top_p,
    )

In [4]:
messages = [
    {"role": "user", "content": "Who are the largest car manufacturers in 2023? Do they each makeEVs or not?"}
]
print(generate(messages))

content="Here's a list of the largest car manufacturers in 2023, along with their electric vehicle (EV) offerings:\n\n1. **Toyota**:\n   - EVs: Yes, Toyota offers a range of EVs, including the bZ4X, Prius Prime, and the upcoming bZ3.\n   - Notable models: Corolla, Camry, RAV4, Tacoma\n\n2. **Volkswagen**:\n   - EVs: Yes, Volkswagen has a wide range of EVs, including the ID.4, ID.3, and the upcoming ID.5.\n   - Notable models: Golf, Jetta, Passat, Tiguan\n\n3. **General Motors (GM)**:\n   - EVs: Yes, GM offers a range of EVs, including the Chevrolet Bolt, Bolt EUV, and the upcoming Chevrolet Silverado EV.\n   - Notable models: Chevrolet Silverado, Equinox, Tahoe, Corvette\n\n4. **" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 200, 'prompt_tokens': 65, 'total_tokens': 265, 'completion_time': 0.474694823, 'completion_tokens_details': None, 'prompt_time': 0.004457535, 'prompt_tokens_details': None, 'queue_time': 0.054703325, 'total_time': 0.479152358}, 'mode

In [5]:
messages = [{"role": "user", "content": "Classify the text into neutral, negative or positive.\nText: I think the food was okay.\nSentiment:"}]
print(generate(messages))

content='The sentiment of the text is neutral. The word "okay" implies a lack of strong emotion or enthusiasm, indicating a neutral opinion about the food.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 68, 'total_tokens': 99, 'completion_time': 0.046882582, 'completion_tokens_details': None, 'prompt_time': 0.004356554, 'prompt_tokens_details': None, 'queue_time': 0.054736165, 'total_time': 0.051239136}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ff181-f0db-7960-a559-98e4b68f50c2-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 68, 'output_tokens': 31, 'total_tokens': 99}


In [6]:
persona = "You are an expert in AI programming assistant.Help solving, writing, explaining any code to make the user's work easy.\n"
instruction= "Give a step by step answer for the user's question.If it's a coding, answer should be working code.\n"
context = "The assistance is used by the some developers.\n"
data_format = """1. Question summery.
2. Explaination
3. Code (if applicabel)
4. Conclusion\n"""
audience = "The user is beginner to intermediate programer.\n"
tone = "Friendly, professional, and concise.\n"
data = "Explain me a C program for sum of two numbers in a simple way with code."
full_prompt = persona + instruction + context + data_format + audience + tone + data
messages = [{"role": "user", "content": full_prompt}]
print(generate(messages,max_new_tokens=350))

content='**Question Summary:**\nThe user wants a C program that calculates the sum of two numbers.\n\n**Explanation:**\nIn C programming, we can use a simple program to calculate the sum of two numbers. This program will take two numbers as input from the user, add them together, and then display the result.\n\n**Code:**\n```c\n#include <stdio.h>\n\nint main() {\n    // Declare variables to store the two numbers\n    int num1, num2;\n\n    // Prompt the user to enter the first number\n    printf("Enter the first number: ");\n    scanf("%d", &num1);\n\n    // Prompt the user to enter the second number\n    printf("Enter the second number: ");\n    scanf("%d", &num2);\n\n    // Calculate the sum of the two numbers\n    int sum = num1 + num2;\n\n    // Display the result\n    printf("The sum of the two numbers is: %d\\n", sum);\n\n    return 0;\n}\n```\n\n**How the code works:**\n\n1. We include the `stdio.h` header file, which provides functions for input/output operations.\n2. We declar

**Chain-of-Thought — zero-shot version**

In [7]:
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]
print(generate(zeroshot_cot_prompt, max_new_tokens=200))

content="Let's break it down step by step:\n\n1. The cafeteria initially had 23 apples.\n2. They used 20 apples to make lunch, so we subtract 20 from 23:\n   23 - 20 = 3\n   Now they have 3 apples left.\n3. Then, they bought 6 more apples. To find the total number of apples they have now, we add 6 to 3:\n   3 + 6 = 9\n\nSo, the cafeteria now has 9 apples." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 80, 'total_tokens': 189, 'completion_time': 0.12580384, 'completion_tokens_details': None, 'prompt_time': 0.014201882, 'prompt_tokens_details': None, 'queue_time': 0.056091907, 'total_time': 0.140005722}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ff181-f455-7660-b04c-baefd62c2d8e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 80, 'output_tokens': 109, 'total_tokens': 

**Tree-of-Thought**

In [8]:
zeroshot_tot_prompt = [
    {"role": "user", "content": (
        "Imagine three different experts are answering this question. "
        "All experts will write down 1 step of their thinking, then share it with the group. "
        "Then all experts will go on to the next step, etc. "
        "If any expert realizes they're wrong at any point then they leave. "
        "The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, "
        "how many apples do they have?' Make sure to discuss the results in short."
    )}
]
print(generate(zeroshot_tot_prompt, max_new_tokens=500))

content="**Expert 1: Math Expert**\nStep 1: The cafeteria initially had 23 apples. They used 20 to make lunch, so I'll subtract 20 from 23 to find out how many apples they have left. 23 - 20 = 3 apples.\n\n**Expert 2: Chef**\nStep 1: I'm thinking about the context of the problem. The cafeteria used 20 apples to make lunch, which means they must have used them for a recipe. I'm not sure how many apples were used in the recipe, so I'll just focus on the initial and final quantities for now.\n\n**Expert 3: Statistician**\nStep 1: I'm thinking about the problem from a statistical perspective. The cafeteria bought 6 more apples, but I'm not sure if this is relevant to the question. I'll set it aside for now and focus on the initial and final quantities.\n\n**Expert 1: Math Expert**\nStep 2: Now that I know they have 3 apples left, I'll add the 6 apples they bought to find the total number of apples. 3 + 6 = 9 apples.\n\n**Expert 2: Chef**\nStep 2: I'm thinking about the context again. If th

**Output validation**

In [9]:
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:
{
  "description": "A SHORT DESCRIPTION",
  "name": "THE CHARACTER'S NAME",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [{"role": "user", "content": one_shot_template}]
output = generate(one_shot_prompt, max_new_tokens=150)
print(output)

content='{\n  "description": "A skilled huntress from a secluded village, trained in the art of archery and survival.",\n  "name": "Eira Shadowglow",\n  "armor": "Leather Tunic and leggings",\n  "weapon": "Longbow and Quiver with 20 arrows, Short sword for close combat"\n}' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 108, 'total_tokens': 179, 'completion_time': 0.149172076, 'completion_tokens_details': None, 'prompt_time': 0.021616122, 'prompt_tokens_details': None, 'queue_time': 0.178084015, 'total_time': 0.170788198}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_03e8423237', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ff181-f96b-7db3-941f-ee07aca91937-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 108, 'output_tokens': 71, 'total_tokens': 179}


**Chain:**

*Multiple chaining*

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}.
Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
 template=template, input_variables=["summary","title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

template = """<s><|user|>
Create a story about {summary} with the title {title}. The main character is:
{character}. Only return the story and it cannot be longer than one paragraph.
<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
 template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")
llm_chain = title | character | story
llm_chain.invoke("a girl that lost her mother")


/tmp/ipykernel_1726/2270005470.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


{'summary': 'a girl that lost her mother',
 'title': '"Beyond the Shadows of Her Smile"',
 'character': '<s><|user|>\nAva Moreno is the 17-year-old protagonist of "Beyond the Shadows of Her Smile," a girl who has been struggling to cope with the loss of her mother, a renowned artist whose paintings seemed to hold secrets and stories of their own. Ava\'s journey is a poignant exploration of grief, identity, and the complexities of a mother-daughter relationship that transcends even death.',
 'story': "Beyond the Shadows of Her Smile\n\nAva Moreno's fingers danced across the canvas, the strokes of her brush echoing the turmoil within her. It had been a year since her mother's passing, but the ache still lingered, a hollowed-out space that seemed to grow larger with each passing day. As she mixed colors on her palette, memories of her mother's studio flooded her mind - the scent of turpentine, the soft hum of the radio, and the whispered secrets that only her paintings seemed to hold. Her

**Memory**: *Windowed Conversation Buffer*

In [11]:
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import LLMChain
template = """<s><|user|>Current conversation:{chat_history}
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
 template=template,
 input_variables=["input_prompt", "chat_history"]
)

memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")
llm_chain = LLMChain(
 prompt=prompt,
 llm=llm,
 memory=memory
)
llm_chain.predict(input_prompt="Hi! My name is Manoj and I am 21 years old.What is 133+ 1?")
print("\n")
llm_chain.predict(input_prompt="I forgot my name. Do you know me")

/tmp/ipykernel_1726/1070673075.py:11: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")


"Nice to meet you, Manoj. You're 21 years young. Now, let's do the math: 133 + 1 = 134.\n\nAs for your question, I do know you, Manoj. You're the 21-year-old I was just chatting with."

2*Conversation Summary*

In [12]:
from langchain_classic.memory import ConversationSummaryMemory

summary_prompt_template = """<s><|user|>Summarize the conversations and update
with the new lines.
Current summary:
{summary}
new lines of conversation:
{new_lines}
New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
 input_variables=["new_lines", "summary"],
 template=summary_prompt_template
)
memory = ConversationSummaryMemory(
 llm=llm,
 memory_key="chat_history",
 prompt=summary_prompt
)
llm_chain = LLMChain(
 prompt=prompt,
 llm=llm,
 memory=memory
)
llm_chain.invoke({"input_prompt": "Hi! My name is Manoj S. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

/tmp/ipykernel_1726/3743928128.py:15: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(


{'input_prompt': 'What was the first question I asked?',
 'chat_history': "There is no conversation to summarize yet. The conversation just started with your request to summarize and update the conversation. \n\nHowever, I can update the conversation with the new lines as you provided:\n\nCurrent summary:\nNo summary yet.\n\nNew lines of conversation:\nHuman: Hi! My name is Manoj S. What is 1 + 1?\nAI: Nice to meet you, Manoj S. The answer to 1 + 1 is 2.\n\nIf you'd like to continue the conversation, I can update the summary accordingly.",
 'text': 'Your first question was to ask me to summarize and update the conversation, but since there was no conversation to summarize, I provided a starting point for the conversation.'}

**Agent tools**:

In [15]:
from langchain_classic.agents import load_tools, Tool,AgentExecutor,create_react_agent
from langchain_community.tools import DuckDuckGoSearchResults

react_template = """Answer the following questions as best you can. You have
access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}"""
prompt = PromptTemplate(
 template=react_template,
 input_variables=["tools", "tool_names","input", "agent_scratchpad"]
)
search = DuckDuckGoSearchResults()
search_tool =Tool(
 name="duckduck",
 description="A web search engine. Use this to as a search engine for general queries.",
 func=search.run,
)
tools=load_tools(["llm-math"],llm=llm)
tools.append(search_tool)
agent = create_react_agent(llm, tools, prompt)
agent_executor= AgentExecutor(agent=agent,tools=tools ,verbose=True, handle_parsing_errors=True,max_iterations=6,max_execution_time=90)
agent_executor.invoke({"input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD."})



> Entering new AgentExecutor chain...
Thought: To find the current price of a MacBook Pro in USD, I should first check the current market price. However, I don't have direct access to real-time market data. I can use a web search engine to find the current price.

Action: duckduck
Action Input: "current price of MacBook Pro in USD"snippet: Get a new MacBook Pro laptop with M5, M5 Pro, or M5 Max from only $166.58 per month. Built for AI. Up to 24-hour battery life. Buy or lease now at apple.com.Students and educators — save on a new Mac. Get special pricing in the Education Store., title: Buy MacBook Pro - Apple, link: https://www.apple.com/shop/buy-mac/macbook-pro, snippet: Find MacBook laptops designed for speed and efficiency. Explore options with extended battery life and powerful performance., title: Amazon.com: Macbook, link: https://www.amazon.com/macbook/s?k=macbook, snippet: Compare prices and track the best Apple deals on MacBook, iMac and Mac mini models., title: Mac Comput

{'input': 'What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD.',
 'output': 'The current price of a MacBook Pro in USD is $2,499, and the price in EUR is approximately €2,134.15, given an exchange rate of 0.85 EUR for 1 USD.'}

**Sematic search**

In [4]:
text = """Interstellar is a 2014 epic science fiction film..."""
texts = text.split('.')
texts = [t.strip(' \n') for t in texts if t.strip(' \n')]

In [5]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeds = embedder.encode(texts, convert_to_numpy=True)
print(embeds.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(1, 384)


In [6]:
import faiss

dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
print(index.is_trained)
index.add(np.float32(embeds))

True


In [8]:
def search(query, number_of_results=3):
    query_embed = embedder.encode([query], convert_to_numpy=True)
    distances, similar_item_ids = index.search(np.float32(query_embed), number_of_results)
    texts_np = np.array(texts)
    results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]], 'distance': distances[0]})
    print(f"Query: '{query}'\nNearest neighbors:")
    return results

In [9]:
query = "how precise was the science"
results = search(query)
results

Query: 'how precise was the science'
Nearest neighbors:


,texts,distance
0,Interstellar is a 2014 epic science fiction film,1.677418e+00
1,Interstellar is a 2014 epic science fiction film,3.402823e+38
2,Interstellar is a 2014 epic science fiction film,3.402823e+38
